# EDA Simple Class

## Steps :

### 1. Data Ingestion
### 2. Data Inspection
### 3. Data Preprocessing
### 4. Data Visualization
### 5. Statistical Reporting



## 1. Data Ingestion / Loading / Pipelining / Accessing

In [1]:
# step 0 : importing library for EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# take the data path from file and load it to pandas dataframe

df1 = pd.read_csv('/content/Weather_data.csv')
df1.head()

In [ ]:
# load iris.xlsx file to df2

df2 = pd.read_excel('/content/Iris.xlsx', engine = 'openpyxl')
df2.head()

In [ ]:
# loading an API based Kaggle dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("velvetcrystal/fifa-world-cup-2026-complete-tournament-statistics")

print("Path to dataset files:", path)

In [ ]:
# copy the files from the 'path' to /content/
!cp -r "$path"/* /content/

In [ ]:
# load the df3 with world cup data from kaggle
df3 = pd.read_csv('//content/world_cup_2026_matches.csv')
df3.head()

In [ ]:
## final dataset for our work

df = pd.read_csv('/content/HousePrices.csv')
df.head()

In [ ]:
# shape of the dataset
df.shape

In [ ]:
# check for all columns
df.columns

In [ ]:
df.dtypes

## Step 2 : Data Inspection

In [ ]:
df.info()

Observations:


In [ ]:
# numerical descriptive stats summary
df.describe()

In [ ]:
# non - numerical statistical summary
df.describe(exclude='number')

In [ ]:
df.describe(include = 'all')

In [ ]:
# null values
df.isnull().sum()

In [ ]:
# Number of unique data in all columns
df.nunique()

In [ ]:
# top 10 cities count
df.city.value_counts().head(10)

### Observations:
1.  The dataset contains 4600 rows and 18 columns, indicating a moderately sized dataset for analysis.
2.  There are no missing values in any of the columns, which simplifies the data cleaning process significantly.
3.  The 'date' column is currently of `object` type and needs to be converted to `datetime` for time-series analysis.
4.  The 'price', 'bedrooms', and 'bathrooms' columns have a minimum value of 0, which could indicate data entry errors or specific property types (e.g., vacant land, studios) and requires further investigation.
5.  Columns like 'waterfront', 'view', and 'condition' are represented as numerical integers but are inherently categorical, which is important for feature engineering.
6.  The 'yr_renovated' column has a minimum value of 0, suggesting that many properties have not been renovated, or the value 0 represents 'no renovation'.
7.  The 'country' column has only one unique value ('USA'), making it a constant column that can be dropped if the analysis is focused within the USA.
8.  The 'street' column has a very high number of unique values (4525 out of 4600 rows), indicating high cardinality, which might be challenging for direct use in some models.
9.  The 'city' data shows a high concentration in a few major cities, especially Seattle, which accounts for a large portion of the entries.

## Step 3: Data Preprocessing

In [ ]:
# Conversions

# Date column to datetime
df['date'] = pd.to_datetime(df['date'])


In [ ]:
# convert yr_built and yr_renovated to datetime(year)
df['yr_built'] = pd.to_datetime(df['yr_built'], format='%Y')
df['yr_renovated'] = df['yr_renovated'].replace(0, pd.NaT)
df['yr_renovated'] = pd.to_datetime(df['yr_renovated'], format='%Y')

In [ ]:
# create categorical columns for waterfront
df['waterfront'] = df['waterfront'].astype('category')

In [ ]:
# drop columns which are not required
# country, street
df.drop(['country', 'street'], axis=1, inplace = True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# mapping waterfront to 0:no, 1:yes
df.waterfront = df.waterfront.map({0:'No', 1:'Yes'})
df.head()

In [ ]:
# Create useful analytics columns
df.date.value_counts()

In [ ]:
# create a age column for property listed
df['age'] = df.date.dt.year - df.yr_built.dt.year
df.head()

In [ ]:
# if house is renovated or not
df['is_renovated'] = np.where(df.yr_renovated.isna(), 'No', 'Yes')
df.head()

In [ ]:
# total area = all sq_ft columns sum
df['total_area'] = df[['sqft_above', 'sqft_basement', 'sqft_living', 'sqft_lot']].sum(axis=1)
df.head()

In [ ]:
# new column for price categories
df['price_category'] = pd.cut(df.price, bins=[0, 200000, 500000, 1000000, 2000000, 5000000, float('inf')],
                              labels=['0-200k', '200k-500k', '500k-1M', '1M-2M', '2M-5M', '5M+'])
df.head()

## Step 4: Data Visaulization

- Univariate Analysis - Distribution
- Bivariate Analysis - Relationship
- Multivariate Analysis - Target - Features

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df.price, bins=50, kde=True, color='red')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Pandas based visualization
# Price Distribution
df.price.plot(kind = 'hist',bins=300, figsize = (10,5), title = 'Price Distribution')


In [ ]:
# price category distribution
df.price_category.value_counts().plot(kind='bar', title='Price Category Distribution')
plt.show()

In [ ]:
# price by cities
df.groupby('city')['price'].mean().sort_values(ascending=False).head(10).plot(kind='bar', title='Average Price by City')


In [ ]:
# time series data
df.groupby(df.yr_built.dt.year)['price'].mean().plot(kind='line', title='Average Price Over Time')

### Observations from Data Preprocessing and Visualization:

**Data Preprocessing (Step 3):**
- **Date and Year Conversions:** The `date` column was successfully converted to datetime format. The `yr_built` and `yr_renovated` columns were also converted to datetime, with `0` values in `yr_renovated` appropriately handled by replacing them with `NaT` to indicate no renovation.
- **Categorical Feature Engineering:** The `waterfront` column was converted to a categorical type and mapped to meaningful 'No'/'Yes' labels.
- **Feature Selection:** Irrelevant columns such as `country` (single unique value) and `street` (high cardinality) were dropped to simplify the dataset and avoid potential noise.
- **New Feature Creation:** Several new analytical features were created:
    - `age`: Calculated as the difference between the listing year and the built year, providing a direct measure of property age.
    - `is_renovated`: A binary categorical feature indicating whether a property has been renovated (`Yes`) or not (`No`), derived from `yr_renovated`.
    - `total_area`: A combined metric summing various square footage columns, offering a holistic view of property size.
    - `price_category`: A categorical variable created by binning the `price` column, which helps in understanding price segments.

**Data Visualization (Step 4):**
- **Price Distribution:** Histograms and KDE plots revealed a highly right-skewed distribution for property prices. This indicates that a large number of properties are concentrated at the lower end of the price spectrum, with a long tail extending to a few very expensive properties. This suggests the presence of outliers and potential need for transformation in modeling.
- **Price Category Distribution:** The bar chart of `price_category` distribution confirmed the skewed nature, with the majority of properties falling into the '200k-500k' and '500k-1M' brackets. This highlights the most common price ranges in the dataset.
- **Average Price by City:** The bar plot showing average price by city clearly indicated significant disparities in property values across different cities. Cities like Medina, Clyde Hill, and Yarrow Point exhibit substantially higher average prices, suggesting that location is a strong determinant of property value.
- **Average Price Over Time (by Year Built):** The line plot of average price by `yr_built` showed fluctuations over time, indicating periods of higher or lower average prices for properties built in specific years. This can reflect economic trends, changes in building standards, or desirability of certain architectural periods.

## Step 5 : Statistical Reporting

### Executive Summary: Data Analysis Progress Report

#### Objective:
To perform Exploratory Data Analysis (EDA) on the House Prices dataset, encompassing data ingestion, inspection, preprocessing, and visualization, to gain insights into the dataset and prepare it for advanced analytics or predictive modeling.

#### Key Findings:

1.  **Data Ingestion & Overview:**
    *   The primary dataset `df` (HousePrices.csv) was successfully loaded, comprising 4600 entries and 18 initial columns. Auxiliary datasets (`Weather_data.csv`, `Iris.xlsx`, `world_cup_2026_matches.csv`) were also ingested for context or demonstration.
    *   No missing values were identified in the `df` dataset, simplifying initial cleaning efforts.

2.  **Initial Data Inspection:**
    *   The `df` dataset's columns included a mix of numerical (e.g., `price`, `sqft_living`) and categorical/object (e.g., `date`, `street`, `city`) types. Critical observations included: the `date` column was an object type needing conversion; `price`, `bedrooms`, and `bathrooms` had minimum values of zero, indicating potential anomalies or specific property types; and `country` was a constant column ('USA').

3.  **Data Preprocessing (Transformations & Feature Engineering):**
    *   **Type Conversions:** `date`, `yr_built`, and `yr_renovated` columns were successfully converted to datetime objects. Special handling was implemented for `yr_renovated` to convert `0` values to `NaT` (Not a Time), correctly indicating unrenovated properties.
    *   **Feature Simplification:** The `country` and `street` columns were dropped due to their lack of variance or high cardinality, respectively, to streamline the dataset.
    *   **Categorical Mapping:** The `waterfront` numerical column was explicitly converted to a categorical type and mapped to 'No'/'Yes' for better interpretability.
    *   **New Feature Creation:** Several value-added features were engineered:
        *   `age`: Property age at the time of listing.
        *   `is_renovated`: A binary indicator for renovation status.
        *   `total_area`: An aggregate measure of property square footage.
        *   `price_category`: Categorical bins for property prices, aiding in segment analysis.

4.  **Data Visualization (Exploratory Analysis):**
    *   **Price Distribution:** Visualizations (histograms, KDE plots) consistently showed a highly right-skewed distribution for property prices, with a majority of properties in lower to mid-price ranges and a long tail extending to very high-value properties. This suggests the presence of outliers and potential need for data transformation for modeling.
    *   **Price Categorization:** Analysis of `price_category` confirmed the dominance of properties in the '200k-500k' and '500k-1M' brackets.
    *   **Geographic Price Variation:** Bar plots demonstrated significant average price differences across cities, highlighting 'Medina', 'Clyde Hill', and 'Yarrow Point' as having substantially higher average property values, underscoring the importance of location.
    *   **Temporal Price Trends:** A line plot of average price by `yr_built` showed fluctuations, indicating historical trends in property values based on construction year.

#### Conclusion:
The dataset has been successfully ingested, inspected, cleaned, and enriched with new features. Initial visualizations have provided crucial insights into price distribution, geographical influences, and historical trends. The data is now in a more structured and informative state, ready for deeper statistical analysis and predictive model development.

### Future Tasks

#### For Step 3: Data Preprocessing (Further Refinements)
1.  **Outlier Management:** Systematically identify and handle outliers in numerical features like `price`, `bedrooms`, `bathrooms`, `sqft_living`, `sqft_lot`, etc., as indicated by the skewed distributions and '0' values.
2.  **Feature Scaling:** Apply appropriate scaling techniques (e.g., StandardScaler, MinMaxScaler) to numerical features to ensure they contribute equally to model training, especially for distance-based algorithms.
3.  **Categorical Encoding:** Encode remaining categorical features (`city`, `statezip`, `waterfront`, `is_renovated`) into numerical representations suitable for machine learning models (e.g., One-Hot Encoding for nominal categories, Ordinal Encoding for ordinal categories if applicable).
4.  **Transformation for Skewed Data:** Consider applying transformations (e.g., logarithmic transformation) to the `price` column and other highly skewed numerical features to normalize their distribution, which can improve model performance.
5.  **Interaction Features:** Explore creating interaction terms between existing features that might capture more complex relationships (e.g., `sqft_living` * `floors`).

#### For Step 5: Statistical Reporting
1.  **Correlation Analysis:** Generate a correlation matrix and heatmap for all numerical features (including newly created ones) to quantitatively assess relationships with the target variable (`price`) and among themselves. Highlight highly correlated features.
2.  **Statistical Tests for Categorical Features:** Perform statistical tests (e.g., ANOVA for `price` vs. `waterfront`, `is_renovated`, `city` categories) to confirm the significance of observed differences in average prices across different categories.
3.  **Detailed Anomaly Investigation:** Conduct a deeper dive into instances where `bedrooms` or `bathrooms` are `0`, determining if these are data entry errors or represent valid property types (e.g., commercial properties without bedrooms, or properties with shared/communal bathrooms).
4.  **Distribution of Other Key Features:** Provide statistical summaries and visualizations (histograms, box plots) for `bedrooms`, `bathrooms`, `floors`, `view`, `condition`, and `age` to understand their individual distributions and potential impact on `price`.